# Spektralno klasterovanje i segmentacija slika pomoću grafova

## Uvod

Ova sveska predstavlja teorijski uvod u projekat. Cilj je da se objasni šta je to spektralno klasterovanje, zašto je korisno, i kako se matematički izvodi — pre nego što pređemo na implementaciju i eksperimente u narednim sveskama. :)

### Problem sa klasičnim klasterovanjem

Klasični algoritmi klasterovanja, poput K-means, pretpostavljaju da su klasteri u obliku konveksnih skupova. U praksi, podaci često imaju mnogo složenije oblike. K-means u takvim slučajevima ne uspeva da pronađe ispravne klastere, jer pokušava da minimizuje rastojanje tačaka od centara klastera, što prirodno favorizuje konveksne regione.

**Spektralno klasterovanje** rešava ovaj problem tako što ne posmatra sirove koordinate tačaka, već **strukturu povezanosti** između njih, predstavljenu kroz graf.

## Osnovna ideja

Umesto direktnog klasterovanja podataka, spektralno klasterovanje prolazi kroz sledeće korake:

1. Podaci se predstave kao **graf** — svaka tačka je čvor, a grane povezuju slične tačke, sa težinom koja odražava jačinu sličnosti.
2. Iz grafa se konstruiše **Laplasijanova matrica**, koja opisuje strukturu povezanosti.
3. Izračunaju se **sopstveni vektori** te matrice — oni otkrivaju prirodne grupe (klastere) unutar grafa.
4. Na dobijenim sopstvenim vektorima (u novom, nižedimenzionom prostoru) primeni se standardni K-means algoritam.

Ključna ideja: klasterovanje se ne radi nad originalnim podacima, već nad njihovom strukturom povezanosti, što omogućava hvatanje mnogo složenijih oblika klastera nego što to klasični algoritmi mogu.

## Matrica sličnosti (W)

Prvi korak je konstrukcija **matrice sličnosti** $W$, gde $W_{ij}$ predstavlja meru sličnosti između tačaka $i$ i $j$. U ovom projektu koristi se Gausov (RBF) kernel:

$$
W_{ij} = \exp\left(-\frac{\|x_i - x_j\|^2}{2\sigma^2}\right)
$$

gde je $\sigma$ parametar koji kontroliše koliko brzo sličnost opada sa rastojanjem (širina Gausovog zvona).

### Zašto baš Gausov kernel?

- **Lokalnost** — zbog eksponencijalnog opadanja sličnosti sa rastojanjem, veze između udaljenih tačaka postaju zanemarljive, dok se zadržavaju veze između lokalno bliskih tačaka.
- **Kontrola preko $\sigma$** — parametar $\sigma$ direktno određuje "širinu susedstva": manje $\sigma$ znači da se samo veoma bliske tačke smatraju sličnim, dok veće $\sigma$ čini graf gušćim i manje osetljivim na lokalnu strukturu.
- **Matematička pogodnost** — Gausov kernel je pozitivno semi-definitan (Mercer-ov uslov), što garantuje da će Laplasijan imati realne, nenegativne sopstvene vrednosti.
- **Standard u literaturi** — koriste ga i Ng, Jordan i Weiss (2002) i Von Luxburg (2007), što omogućava lakše poređenje rezultata sa poznatim radovima.

Parametar $\sigma$ je jedan od najvažnijih hiperparametara u celom postupku, i njegov uticaj na rezultate detaljno se istražuje u narednim sveskama.

## Matrica stepena (D) i Laplasijan (L)

### Matrica stepena D

$D$ je dijagonalna matrica gde je:

$$
D_{ii} = \sum_j W_{ij}
$$

Dakle, $D_{ii}$ predstavlja koliko je čvor $i$ ukupno povezan sa ostatkom grafa — sumu jačina svih njegovih veza.

### Laplasijan L = D − W

Laplasijanova matrica se dobija kao $L = D - W$. Njena ključna osobina vidi se kada se primeni na proizvoljan vektor $x$:

$$
(Lx)_i = \sum_j W_{ij}(x_i - x_j)
$$

Ovaj izraz meri razliku vrednosti $x_i$ u odnosu na vrednosti njegovih jako povezanih suseda. Ukoliko su vrednosti $x$ slične za povezane čvorove, $Lx$ je blizu nule, što znači da je funkcija glatka na grafu. Zbog toga Laplasijan meri **promenu**, odnosno glatkoću funkcije definisane nad grafom.

### Zašto su bitni sopstveni vektori matrice L
Za sopstveni vektor $v$ i sopstvenu vrednost $\lambda$ važi $Lv = \lambda v$. Sopstvena vrednost određuje stepen promene sopstvenog vektora duž grafa: manje vrednosti odgovaraju glatkijim funkcijama.

- Najmanja sopstvena vrednost je 0, a odgovarajući sopstveni vektor je konstantan, što odgovara slučaju kada ceo graf predstavlja jedan klaster.
- Narednih $k$ sopstvenih vektora sa najmanjim sopstvenim vrednostima sadrže informaciju o prirodnoj podeli grafa: čvorovi unutar istog klastera imaju slične vrednosti, dok se vrednosti između različitih klastera razlikuju.

Postavljanjem ovih vektora kao kolona nove matrice, podaci se preslikavaju u novi $k$-dimenzioni prostor u kome su tačke istog klastera blizu. U tom prostoru se zatim može primeniti K-means, čak i kada klasteri u originalnom prostoru imaju složen, nekonveksan oblik.

## Normalizovani Laplasijan

Pored osnovnog (nenormalizovanog) Laplasijana $L = D - W$, u praksi se često koristi **simetrični normalizovani Laplasijan**:

$$
L_{sym} = D^{-1/2} L D^{-1/2}
$$

Ova normalizacija (koju uvode Ng, Jordan i Weiss, 2002) kompenzuje razlike u stepenu povezanosti pojedinih čvorova — normalizacija skalira uticaj svakog čvora u odnosu na njegov stepen, čime se smanjuje pristrasnost prema gusto povezanim delovima grafa.

Ovo je posebno bitno kada klasteri variraju u veličini ili gustini — čest slučaj kod segmentacije slika, gde postoje veliki homogeni regioni (npr. nebo) i manji, takodje detaljni regioni (npr. ivice objekata). Bez normalizacije, gusto povezani regioni bi dominirali sopstvenom-dekompozicijom samo zbog svoje veličine, ne zbog stvarne strukture.

## Koraci algoritma — pregled

Kompletan algoritam spektralnog klasterovanja (Ng-Jordan-Weiss varijanta) sastoji se od sledećih koraka:

1. Na osnovu ulaznih podataka $X$ konstruiše se matrica sličnosti $W$, najčešće korišćenjem Gausovog kernela.
2. Iz matrice sličnosti računaju se matrica stepena $D$ i Laplasijan (nenormalizovan $L$ ili normalizovan $L_{sym}$).
3. Izračuna se $k$ najmanjih sopstvenih vektora Laplasijana i formira se matricu $U \in \mathbb{R}^{n \times k}$ čije su kolone izračunati sopstveni vektori.
4. Svaki red matrice $U$ posmatra se kao nova $k$-dimenziona reprezentacija odgovarajuće tačke.
5. Primenjuje se standardni algoritam K-means nad redovima matrice $U$, čime se dobijaju konačne oznake klastera.

Implementacija ovih koraka nalazi se u `src/spectral.py`, a testiranje na sintetičkim podacima je predmet naredne sveske (`02_synthetic_data.ipynb`).

## Primena na segmentaciju slika

Segmentacija slike podrazumeva podelu slike na regione na osnovu vizuelne sličnosti (boja, tekstura, pozicija), bez nužnog prethodnog znanja o tome šta ti regioni predstavljaju.

Ideja je da se svaki piksel (ili grupa piksela) tretira kao čvor grafa, gde su grane određene sličnošću u boji i blizinom u prostoru. Spektralno klasterovanje se zatim primenjuje da grupiše piksele u smislene segmente.

**Praktičan izazov**: slika dimenzija $n \times n$ piksela dovodi do matrice sličnosti veličine $n^2 \times n^2$, što sopstvenu-dekompoziciju čini računski veoma skupom. U ovom projektu, ovaj problem se rešava grupisanjem piksela u **superpiksele** (npr. SLIC algoritam) pre primene spektralnog klasterovanja. Ovaj deo je detaljnije obrađen u `03_image_segmentation.ipynb`.

## Literatura

1. [Ng, A., Jordan, M., & Weiss, Y. (2002). *On Spectral Clustering: Analysis and an Algorithm.*](https://www.ee.columbia.edu/~dpwe/papers/NgJW01-specclus.pdf)
2. [Von Luxburg, U. (2007). *A Tutorial on Spectral Clustering.*](https://arxiv.org/pdf/0711.0189)
3. [Shi, J., & Malik, J. (2000). *Normalized Cuts and Image Segmentation.*](https://www.cs.cmu.edu/~jshi/papers/pami_ncut.pdf)